# 11장 : 종양 탐지를 위한 분류 모델 훈련

 + 파이토치 DataLoader로 데이터 로딩하기
 + CT 데이터에서 분류를 수행하는 모델 만들기
 + 애플리케이션 기본 구조 설정
 + 메트릭 로깅과 표시

앞서 0장에서는 암 진단 프로젝트를 위한 기본 준비를 마쳤다. 폐암에 대한 의학적인 세부 내용도 파악했고, 프로젝트에서 사용할 주요 데이터 소스도 살펴본 후 원본 CT 스캔을 파이토치 `Dataset` 인스턴스로 변환해봤다. 데이터셋이 준비되었으니 손쉽게 훈련 데이터를 사용할 수 있다. 계속 이어서 진행해본다.

## 11.1 기본 모델과 훈련 루프

이번 11장에서는 크게 두 가지를 진행한다. 먼저 2부의 나머지 부분에서 더 큰 프로젝트를 탐험하기 위한 도구가 될 분류 모델과 훈련 루프를 만들 예정이다. 이를 위해 10장에서 만든 `Ct` 클래스와 `LunaDataset` 클래스를 `DataLoader` 인스턴스에 넣는다. 그리고 다음 단계로, 이 인스턴스를 데이터와 함께 훈련 루프와 검증 루프를 거쳐 분류 모델에 입력한다.

그 다음은 2부에서 가장 어려운 과제로, 훈련 루프 실행 결과를 사용하기 이해 여러 제약이 있는 지저분한 데이터를 어떻게 고품질의 결과로 만드는지를 설명한다. 이후 장에서는 데이터의 제약 사항을 어떻게 완화할지 구체적인 방안을 살펴본다.

구현의 기본 구조는 다음과 같다.

  - 모델을 초기화하고 데이터를 로딩한다.
  - 어느 정도 임의로 선택한 에포크 수로 루프를 반복한다.
    - `LunaDataset`이 반환한 훈련 데이터의 배치 루프를 돈다.
    - 백그라운드에서 데이터로더 워커(worker) 프로세스는 적합한 배치를 읽어들인다.
    - 배치(batch)를 분류 모델에 전달하여 결과를 얻는다.
    - 추정 결과를 실측 데이터와 비교하여 손실을 계산한다.
    - 임시 데이터 구조에 모델의 성능 메트릭을 기록한다.
    - 오차 역전파로 모델 가중치를 조정한다.
    - (훈련 루프와 유사하게) 검증 데이터 배치로 루프 반복
    - (백그라운드 워커 프로세스에서) 검증 데이터 배치를 읽어들인다.
    - 배치를 분류하고 손실을 계산한다.
    - 모델이 검증 데이터에 대해 얼마나 잘 동작했는지를 기록한다.
    - 매 에포크마다 진행 상황과 성능 정보를 출력한다.

이 장의 코드를 따라가며, 1부의 훈련 루프와 크게 두 가지 측면에서 다른 부분에 주목하자. 첫째로, 이전보다 상당히 복잡해진 만큼 구조적으로 프로그램을 작성해야 한다. 그렇지 않으면 코드가 금방 지저분해진다. 프로젝트는 잘 만들어진 여러 함수를 훈련 어플리케이션에서 사용하며, 데이터셋 등을 위한 코드는 자체적인 파이썬 모듈로 분리할 것이다.

이 프로젝트 이후 각자 자신만의 프로젝트를 수행할 때는 복잡도에 어울리는 구조나 설계 수준을 택하도록 한다. 충분한 구조화가 없으면 실험을 깔끔하게 수행하거나 문제 해결하기가 어려우며, 심지어 자신이 무엇을 하는지도 설명하기 쉽지 않다. 반대로, 필요 이상의 구조화는 쓸모없는 인프라 구조 작성에 시간을 낭비하게 될 것이며, 결국 그 틀에 맞추기 위한 여러 작업을 하느라 진행이 더뎌질 것이다. 어떻게 보면 인프라 구조에 시간을 허비하는 건 실제 프로젝트에 필요한 고난도 작업을 기피하는 핑계나 게으름을 부릴 구실이 될 수도 있다. 이런 함정에 빠져서는 안 된다!

1부와 이번 11장의 또 다른 차이점으로, 훈련 진척에 대한 다양한 메트릭을 수집하는 점이 있다. 좋은 메트릭 로깅 없이 훈련에 준 변화가 어떤 영향을 미치는지 정확하게 파악하기란 거의 불가능하다. 다음 12장에도 중요한 내용이 많지만, 이번 11장에서는 단순한 메트릭 수집을 넘어 $\textbf{올바른 메트릭}$ 을 수집하는 것이 얼마나 중요한지에 대해서도 이야기한다.

이번 11장에서는 메트릭을 수집하기 위한 인프라 구조를 깔아 놓는다. 그리고 이 구조를 통해 전체와 개별 클래스 단위로 손실값과 잘 분류된 샘플의 백분율을 수집하고 표시한다. 이 정도 시작이면 충분하며, 추후에 12장에서는 현실적인 실제 메트릭셋을 더 자세히 다룬다.


## 11.2 애플리케이션의 메인 진입점

앞에서 진행한 훈련과의 구조적인 큰 차이는 2부가 완전한 명령형 애플리케이션이라는 점이다. 명령형 인자를 파싱하고 완전한 `--help` 옵션을 제공하며 여러 환경에서 쉽게 실행할 수 있다. 이를 통해 주피터와 배쉬(bash) 쉘에서 손쉽게 실행할 수 있다.

애플리케이션은 클래스로 구현하여 필요할 때 인스턴스로 만들어 실행한다. 클래스이므로 손쉽게 테스트나 디버깅, 다른 파이썬 프로그램에서의 실행 등이 가능하다. 추가로 운영체제 수준에서 별도 프로세스로 띄우지 않아도 애플리케이션을 구동할 수 있다.

함수 호출이나 OS 레벨의 프로세스로 훈련을 구동할 수 있다면 함수 호출을 래핑하여 주피터 노트북에 넣을 수 있으므로, 코드를 쉽게 네이티브 CLI나 브라우저에서 호출할 수 있는 장점이 있다.

In [1]:

from functions.util import importstr
from functions.util import logging

log = logging.getLogger('notebook')

def run(app, *argv):
    argv = list(argv)
    argv.insert(0, f'--num-workers=4')  # <1>
    log.info(f"Running: {app}({argv!r}).main()")
    
    app_cls = importstr(*app.rsplit('.', 1))  # <2>
    app_cls(argv).main()
    
    log.info(f"Finished: {app}.{argv!r}).main()")


표준에 가까운 뻔한 코드가 방해되지 않게 하자. 파일 끝부터 살펴보면 애플리케이션 객체를 인스턴스로 만들고 `main` 메소드를 호출하는 표준 형태의 `if main` 코드를 볼 수 있다.

```python
if __name__ == '__main__':
    LunaTrainingApp().main()
```

파일에서 처음의 애플리케이션 클래스를 보면 방금 호출한 두 함수 `__init__`과 `main`을 볼 수 있다. 명령행 인자를 받으므로 애플리케이션 `__init__` 함수에서 표준 `argparse` 라이브러리를 사용한다. 초기화 함수에 커스텀 인자를 전달할 수 있고 그래야만 한다. `main` 메소드는 애플리케이션의 핵심 로직을 위한 진입점이다.


```python
class LunaTrainingApp:
    def __init__(self, sys_argv=None):
        if sys_argv is None:
            sys_argv = sys.argv[1:]

        parser = argparse.ArgumentParser()
        parser.add_argument(
            "--num-workers",
            help="Number of worker processes for background data loading",
            default=8,
            type=int,
        )
        # ...63행
        self.cli_args = parser.parse_args(sys_argv)
        self.time_str = datetime.datetime.now().strftime("%Y-%m-%d_%H.%M.%S")

        self.trn_writer = None
        self.val_writer = None
        self.totalTrainingSamples_count = 0

        self.use_cuda = torch.cuda.is_available()
        self.device = torch.device("cuda" if self.use_cuda else "cpu")

        self.model = self.initModel()
        self.optimizer = self.initOptimizer()

    # ...137행
    def main(self):
        log.info(f"Starting {type(self).__name__}, {self.cli_args}")
```

이 구조는 매우 일반적이라 향후 다른 프로젝트에서 재사용 가능하다. `__init__` 안에서의 인자 파싱은 애플리케이션 호출에 영향을 받지 않고 설정할 수 있다.

## 11.3 사전 훈련 설정과 초기화

에포크 내의 각 배치를 순회하기에 앞서, 초기화가 필요하다. 인스턴스로 만들지도 않고 훈련시킬 수는 없는 노릇이다. 두 가지를 해야 한다.

  1. 모델과 옵티마이저의 초기화
  2. `Dataset`과 `DataLoader` 인스턴스 초기화

`LunaDataset`은 랜덤으로 선택한 샘플셋을 정의하여 훈련 에포크를 채워줄 것이고 `DataLoader` 인스턴스는 데이터셋으로부터 데이터를 읽는 작업을 수행하여 애플리케이션에 제공한다.

### 11.3.1 모델과 옵티마이저 초기화

이 절에서는 `LunaModel` 세부를 블랙박스처럼 생각한다. 내부 동작에 대해서는 11.4절에서 자세히 다룬다.

```python
class LunaTrainingApp:
    def __init__(self, sys_argv=None):
        # ... 70행
        self.use_cuda = torch.cuda.is_available()
        self.device = torch.device("cuda" if self.use_cuda else "cpu")

        self.model = self.initModel()
        self.optimizer = self.initOptimizer()

    def initModel(self):
        model = LunaModel()
        if self.use_cuda:
            log.info(f"Using CUDA; {torch.cuda.device_count()} devices.")
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(self.device)
        return model

    def initOptimizer(self):
        return SGD(self.model.parameters(), lr=0.001, momentum=0.99)
        # return Adam(self.model.parameters())
```

훈련을 위해 두 개 이상의 GPU를 사용한다면 `nn.DataParallel` 클래스를 사용하여 모든 GPU에 작업을 분산해서 처리 후 파라미터를 다시 모아 조정 작업을 재동기화하는 방식으로 진행한다. 이러한 동작은 모델 구현이나 모델을 사용하는 코드와 완전히 투명하게 분리되어 있다.

$\textbf{DataParallel과 DistributedDataParallel}$
> 이 책에서는 여러 GPU를 활용하기 위해 `DataParallel`을 사용한다. 우리가 `DataParallel`을 선택한 이유는 모델을 집어넣기만 하면 되는 식의 단순한 래퍼이기 때문이다. 그러나 복수 개의 GPU를 사용할 때 최상의 성능을 내는 방법은 아니며, 단일 머신에서 쓸 수 있는 하드웨어만 사용 가능하다.

> 파이토치는 `DistributedDataParallel`도 제공한다. 이것은 여러 GPU를 사용하거나 둘 이상의 머신에 작업을 분산할 때 추천하는 래퍼 클래스다. 다만 설정과 구성이 단순하지 않으며 세부사항은 ddp_tutorial 확인

`self.use_cuda` 값이 `True`인 경우라면 `self.model.to(device)` 호출로 모델 파라미터를 GPU로 옮기고 다양한 컨볼루션과 대량의 숫자 연산에 GPU를 사용하도록 설정한다.

모멘텀과 함께 기본 옵티마이저로 확률적 경사 하강을 사용한다. 처음 시작하는 입장에서 옵티마이저를 고른다면 SGD는 안전한 선택이다. SGD로 잘 풀리지 않는 문제도 있긴 하지만 상대적으로 드문 편이다. 마찬가지로, 학습률 0.001이나 모멘텀 0.9도 상당히 안전한 선택에 해당하는 값이다. 경험에 따르면, 이렇게 값을 지정한 SGD는 광범위한 프로젝트에서 꽤 잘 동작하며 처음에 잘 안된다면 학습률을 간단히 0.01이나 0.0001로 바꿔 시도해보는 것도 해볼 만하다.

학습률, 모멘텀, 신경망 규모 등 기타 유사한 설정을 체계적으로 바꿔가며 찾아가는 과정을 $\textbf{하이퍼파라미터 탐색}^{\text{hyperparameter search}}$라고 부른다. 이 외에도 다음 12장에서는 눈에 띄는 여러 문제를 우선해서 다루고자 한다. 이 문제들을 해결하고 난 뒤, 이 값들을 세밀하게 $\text{조정}^{\text{fine-tune}}$할 예정이다. 옵티마이저를 Adam으로 바꿔서 쓸 수도 있다.

### 11.3.2 데이터 로더의 관리와 데이터 공급

앞서 10장에서 만든 `LunaDataset` 클래스는 우리가 확보한 거친 원본 데이터와 파이토치 빌딩 블럭을 위한 구조화된 텐서 사이의 가교 역할을 한다. 예를 들어 `torch.nn.Conv3d` 는 5차원 입력 $(N, C, D, H, W)$ 인 샘플 수와 샘플당 채널 수뿐만 아니라 깊이와 높이, 마지막으로 너비까지 입력값으로 줘야 한다. CT의 3차원과는 완전히 다르다.

`LunaDataset.__getitem__`에 있던 `candidate_t = candidate_t.unsqueeze(0)` 는, 우리가 사용하는 데이터의 네 번째 차원인 '채널'을 제공한다. 4장의 RGB 이미지에는 빨강, 녹색, 파랑에 대해 각각 세 개의 채널이 있었따. 천문학 데이터에는 감마선, 엑스선, 자외선, 가시광선, 적외선, 극초단파, 전파 등의 다양한 전자기 스펙트럼이 있으며, 각 전자기당 수십 개의 데이터가 있을 수 있다. CT 스캔은 단일한 밀도를 가지므로 채널 차원의 크기는 1이다.

5차원 $N$ 은 배치 차원으로, 동일한 배치 내에서 샘플을 구별하는 용도로 사용한다. 파이토치 `DataLoader` 클래스가 대신 구현해주기 때문에 배치를 직접 구현할 필요는 없다. 이미 CT 스캔을 파이토치 텐서로 바꾸기 위해 `LunaDataset` 클래스를 만들었으니 데이터셋을 데이터 로더에 연결하기만 하면 된다.


```python
    def initTrainDl(self):
        train_ds = LunaDataset(
            val_stride=10,
            isValSet_bool=False,
        )

        batch_size = self.cli_args.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()

        train_dl = DataLoader(
            train_ds,
            batch_size=batch_size,
            num_workers=self.cli_args.num_workers,
            pin_memory=self.use_cuda,
        )

        return train_dl

    def initValDl(self):
        val_ds = LunaDataset(
            val_stride=10,
            isValSet_bool=True,
        )

        batch_size = self.cli_args.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()

        val_dl = DataLoader(
            val_ds,
            batch_size=batch_size,
            num_workers=self.cli_args.num_workers,
            pin_memory=self.use_cuda,
        )

        return val_dl

    def initTensorboardWriters(self):
        if self.trn_writer is None:
            log_dir = os.path.join("runs", self.cli_args.tb_prefix, self.time_str)

            self.trn_writer = SummaryWriter(
                log_dir=f"{log_dir}-trn_cls-{self.cli_args.comment}"
            )
            self.val_writer = SummaryWriter(
                log_dir=f"{log_dir}-val_cls-{self.cli_args.comment}"
            )

    def main(self):
        log.info(f"Starting {type(self).__name__}, {self.cli_args}")

        train_dl = self.initTrainDl()
        val_dl = self.initValDl()
```

데이터 로더는 개별 샘플을 배치로 만들 뿐만 아니라, 별도의 프로세스와 공유 메모리를 사용한 병렬 로딩도 제공한다. 데이터 로더 인스턴스를 만들 때 `num_workers=...`만 지정해주면 나머지는 알아서 처리되고, 각 워커 프로세스는 배치를 만들어낸다(내가 직접 해야 되는게 아니라 데이터 로더가 알아서 다한다).

그리고 코드에서
```python
        train_progress = tqdm(train_dl, desc="E{} Training".format(epoch_ndx), total=len(train_dl))
        for batch_ndx, batch_tup in enumerate(train_progress):
```
처럼 루프를 돌 때 매번 `Ct`를 로드하고 샘플을 가져와 배치할 때까지 기다릴 필요가 없다. 이미 로딩이 끝난 `batch_tup`을 바로 사용하며 워커 프로세스는 다음 순회 시 사용하기 위한 또 다른 배치를 로드하기 위해 자동으로 해제될 것이다. 파이토치의 데이터 로딩 기능을 사용하면 GPU 계산과 데이터 로딩을 동시에 진행하므로 대부분의 경우 프로젝트를 빠르게 실행할 수 있다.

## 11.4 첫 번째 경로 신경망 설계

종양 탐지 컨볼루션 신경망의 설계 범위는 사실상 무한하다. 다행히도 사람들은 지난 십여 년 동안 이미지 인식에 대해 효과적인 모델을 연구해왔다. 대부분이 2차원 이미지에 대한 연구지만 일반적인 아키텍처에 대한 아이디어는 3차원에도 잘 적용할 수 있으므로 테스트가 많이 진행된 신경망 아키텍처를 출발점으로 삼을 수 있다.

우리는 8장에서 사용한 신경망 설계를 기본으로 한다. 입력 데이터가 3차원이므로 모델을 업데이트해야 하고 복잡한 세부 내용을 추가해야 하지만, 기본적으로 그렇다는 것이다. 물론 분류나 세그멘테이션 프로젝트를 더 깊이 진행할수록, 여기에 맞춰 더 많이 개조해야 한다.

### 11.4.1 핵심 컨볼루션

분류 모델에서는 테일, 백본(혹은 바디), 헤드로 구성된 구조가 흔하다.

  - $\textbf{테일}^{\text{tail}}$ : 입력을 신경망에 넣기 전 처리 과정을 담당하는 제일 첫 부분의 일부 계층
     - 이러한 앞 단 계층은 백본이 원하는 형태로 입력을 만들어야 함
     - 이 책에서는 단순 배치 정규화 계층을 사용하지만, 테일에 컨볼루션층이 들어가 있는 경우도 있음
     - 테일에 들어 있는 컨볼루션은 이미지 크기를 공격적으로 다운샘플링하기 위한 용도가 대부분이다(우리의 경우 맨처음에 크롭해서 괜찮다).

  - $\textbf{백본}^{\text{backbone}}$ : 여러 계층을 가지며 일반적으로는 연속된 블럭에 배치된다.
     - 각 블럭은 동일한 세트의 계층을 가지며 블럭과 블럭을 거칠 때마다 필요한 입력 크기나 필터 수가 달라진다.

  - $\textbf{헤드}^{\text{head}}$ : 백본의 출력을 받아 원하는 출력 형태로 바꾼다.
     - 컨볼루션 신경망에서 이 작업은 중간 출력물을 $ \text{평탄화}^{\text{flattening}}$ 하고 $ \text{완전 연결 계층}^{\text{fully connected layer}}$ 에 전달하는 역할을 하기도 한다.
     - 구별할 클래스가 많은 경우에 두 번째 완전 연결 계층이 더욱 적합하기도 하나, 그렇지 않은 경우에도 두 번째 완전 연결 계층으로 전달하기도 한다.
     - 이진분류에서는 복잡하게 만들 필요가 없으므로 하나의 평탄화 계층만 사용한다.

In [2]:
import torch.nn as nn

class LunaBlock(nn.Module):
    def __init__(self, in_channels, conv_channels):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels, conv_channels, kernel_size=3, padding=1, bias=True,
        )
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(
            conv_channels, conv_channels, kernel_size=3, padding=1, bias=True,
        )
        self.relu2 = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool3d(2, 2)

    def forward(self, input_batch):
        block_out = self.conv1(input_batch)
        block_out = self.relu1(block_out)
        block_out = self.conv2(block_out)
        block_out = self.relu2(block_out)

        return self.maxpool(block_out)

### 11.4.2 전체 모델

In [3]:
import math

class LunaModel(nn.Module):
    def __init__(self, in_channels=1, conv_channels=8):
        super().__init__()

        self.tail_batchnorm = nn.BatchNorm3d(1)

        self.block1 = LunaBlock(in_channels, conv_channels) # (N, C, D, H, W) = (N, 1, 32, 48, 48) -> (N, 8, 16, 24, 24)
        self.block2 = LunaBlock(conv_channels, conv_channels * 2) # (N, 8, 16, 24, 24) -> (N, 16, 8, 12, 12)
        self.block3 = LunaBlock(conv_channels * 2, conv_channels * 4) # (N, 16, 8, 12, 12) -> (N, 32, 4, 6, 6)
        self.block4 = LunaBlock(conv_channels * 4, conv_channels * 8) # (N, 32, 4, 6, 6) -> (N, 64, 2, 3, 3)

        self.head_linear = nn.Linear(1152, 2) # (N, C*D*H*W) = (N, 64*2*3*3)
        self.head_softmax = nn.Softmax(dim=1)

        self._init_weights()

    # see also https://github.com/pytorch/pytorch/issues/18182
    def _init_weights(self):
        for m in self.modules():
            if type(m) in {
                nn.Linear, nn.Conv3d, nn.Conv2d, nn.ConvTranspose2d, nn.ConvTranspose3d,
            }:
                nn.init.kaiming_normal_(
                    m.weight.data, a=0, mode='fan_out', nonlinearity='relu',
                )
                if m.bias is not None:
                    fan_in, fan_out = \
                        nn.init._calculate_fan_in_and_fan_out(m.weight.data)
                    bound = 1 / math.sqrt(fan_out)
                    nn.init.normal_(m.bias, -bound, bound)



    def forward(self, input_batch):
        bn_output = self.tail_batchnorm(input_batch)

        block_out = self.block1(bn_output)
        block_out = self.block2(block_out)
        block_out = self.block3(block_out)
        block_out = self.block4(block_out)

        # (N, 64, 2, 3, 3) -> (N, 64*2*3*3)
        conv_flat = block_out.view(
            block_out.size(0),
            -1,
        )
        linear_output = self.head_linear(conv_flat)

        return linear_output, self.head_softmax(linear_output)

테일은 상대적으로 단순하다. 우리는 8장에서 살펴본 `nn.BatchNorm3d` 를 사용해 입력을 정규화한다. 또한 `nn.BatchNorm3d`는 입력값을 이동시키고 비율을 조정해서 평균이 0이고 표준 정규분포 1을 따르도록 만든다. 다소 특히한 하운스필드 단위(HU)는 신경망 뒤에선 더 이상 보이지 않는다. 딱히 이렇게 되도록 만든 이유는 없다. 우리는 입력 단위를 알고 있고, 관련 조직이 가져야 하는 값이 무엇인지도 파악하고 있으므로, 수정된 정규화를 손쉽게 구현할 수 있다. 어떤 방법이 더 나은지는 확실하지 않다.

백본에는 네 개의 반복 블럭이 있고 블럭 구현은 `nn.Module` 서브클래스를 각각 구분하여 사용했다. 각 블럭은 $2 \times 2 \times 2$ 맥스 풀링 연산으로 끝나므로 네 개의 층을 거치면 이미지는 각 차원마다 16배 줄어든 형태가 된다. 10장에서 데이터 덩어리가 $32 \times 48 \times 48$로 주어졌던 것을 상기해보면, 데이터는 백본을 거치며 $2 \times 3 \times 3$ 이 됨을 알 수 있다.

마지막으로 테일에는 완전 연결 계층 뒤로 `nn.Softmax`가 따라온다. 소프트맥스는 단일 레이블 분류에 유용한 함수로, 몇 가지 유용한 특성이 있다. 소프트맥스 함수는 출력을 0과 1 사이의 값으로 묶어서, 입력의 절댓값이 커져도 영향받지 않으며 단지 입력 내에서 상대적인 값에만 영향 받으므로 정답이 얼마나 확실한지 표현하기에 좋다.

In [4]:
# 소프트맥스 단순 구현 예
logits = [1, -2, 3]
exponential_ = [math.e ** x for x in logits]
exponential_

[2.718281828459045, 0.1353352832366127, 20.085536923187664]

In [5]:
softmax = [x / sum(exponential_) for x in exponential_]
softmax

[0.11849965453500959, 0.0058997504019027815, 0.8756005950630876]

> $\textbf{복잡한 문제 : 컨볼루션을 선형으로 바꾸기}$

```python
    def forward(self, input_batch):
        bn_output = self.tail_batchnorm(input_batch)

        block_out = self.block1(bn_output)
        block_out = self.block2(block_out)
        block_out = self.block3(block_out)
        block_out = self.block4(block_out)

        conv_flat = block_out.view(
            block_out.size(0),   # 배치 크기
            -1,
        )
        linear_output = self.head_linear(conv_flat)

        return linear_output, self.head_softmax(linear_output)
```

`self.block4`의 출력은 `(N, 64, 2, 3, 3)` 으로, 완전 연결계층에 전달하려면 `(N, 64*2*3*3)` 으로 평탄화(flatten) 한다. 이 역할을 `view`로 수행할 수 있다. 이러한 평탄화 연산은 상태가 없으므로(동작에 영향을 미치는 파라미터 저장을 요구하지 않으므로) 간단히 `forward`함수 내에서 연산을 실행한다. 

`forward` 메소드는 출력을 위해 $\textbf{로짓}^{\text{logit}}$ 과 소프트맥스로 확률을 만든다. 로짓은 신경망을 타고 만들어지며 소프트맥스 계층을 통해 확률로 만들어지기 전의 숫자값이다. 로짓은 임의의 범위의 실수 값에 해당하지만 소프트맥스를 통과하면 0에서 1까지의 값으로 줄어든다.

훈련 중에는 `nn.CrossEntropyLoss` 계산을 위해 로짓값을 사용하며 샘플을 실제 분류할 때는 확률값을 사용한다(이렇게 하는 데에는 숫자값에 안정성을 더하려는 목적이 있다. 기울기를 전파할 때 지수를 사용해 연산한 결과로 32비트 부동소수점 수를 사용하면 문제를 일으킬 소지가 있기 때문이다). 이런 식으로 훈련 때와 실제 제품으로 사용할 때가 서로 다른 경우는 흔하며 특별히 소프트맥스처럼 상태값이 따로 없고 두 값의 차이가 단순한 경우는 더욱 그렇다.

> $\textbf{초기화}$

모델이 좋은 성능을 낼 수 있도록 동작하려면, 모델의 가중치, 편향값, 여러 파라미터가 특정 속성을 드러내야 한다. 계층의 출력 잘 동작하게 하는 가장 단순한 방법으로 신경망 가중치의 초기화가 있다.


```python
    # see also https://github.com/pytorch/pytorch/issues/18182
    def _init_weights(self):
        for m in self.modules():
            if type(m) in {
                nn.Linear, nn.Conv3d, nn.Conv2d, nn.ConvTranspose2d, nn.ConvTranspose3d,
            }:
                nn.init.kaiming_normal_(
                    m.weight.data, a=0, mode='fan_out', nonlinearity='relu',
                )
                if m.bias is not None:
                    fan_in, fan_out = \
                        nn.init._calculate_fan_in_and_fan_out(m.weight.data)
                    bound = 1 / math.sqrt(fan_out)
                    nn.init.normal_(m.bias, -bound, bound)
```

## 11.5 모델 훈련과 검증

```python
    # ...
    def main(self):
        log.info(f"Starting {type(self).__name__}, {self.cli_args}")

        train_dl = self.initTrainDl()
        val_dl = self.initValDl()

        for epoch_ndx in range(1, self.cli_args.epochs + 1):

            log.info(
                "Epoch {} of {}, {}/{} batches of size {}*{}".format(
                    epoch_ndx,
                    self.cli_args.epochs,
                    len(train_dl),
                    len(val_dl),
                    self.cli_args.batch_size,
                    (torch.cuda.device_count() if self.use_cuda else 1),
                )
            )

            trnMetrics_t = self.doTraining(epoch_ndx, train_dl)
            self.logMetrics(epoch_ndx, "trn", trnMetrics_t)

            valMetrics_t = self.doValidation(epoch_ndx, val_dl)
            self.logMetrics(epoch_ndx, "val", valMetrics_t)

    def doTraining(self, epoch_ndx, train_dl):
        self.model.train()
        trnMetrics_g = torch.zeros(
            METRICS_SIZE,
            len(train_dl.dataset),
            device=self.device,
        )

        train_progress = tqdm(train_dl, desc="E{} Training".format(epoch_ndx), total=len(train_dl))
        for batch_ndx, batch_tup in enumerate(train_progress):
            self.optimizer.zero_grad()

            loss_var = self.computeBatchLoss(
                batch_ndx, batch_tup, train_dl.batch_size, trnMetrics_g
            )

            loss_var.backward()
            self.optimizer.step()

            # # This is for adding the model graph to TensorBoard.
            # if epoch_ndx == 1 and batch_ndx == 0:
            #     with torch.no_grad():
            #         model = LunaModel()
            #         self.trn_writer.add_graph(model, batch_tup[0], verbose=True)
            #         self.trn_writer.close()

        self.totalTrainingSamples_count += len(train_dl.dataset)

        return trnMetrics_g.to("cpu")
```

이 훈련 루프가 이전 장의 훈련 루프와 다른 점은 다음과 같다.

  - `trnMetrics_g` 텐서가 훈련 중에 자세한 클래스 단위 메트릭을 수집한다. 우리 프로젝트처럼 규모가 큰 프로젝트에서는 이 값으로 중요한 통찰을 얻는 경우가 많기 때문에 필요하다.
  - 실제 손실 계산은 `computeBatchLoss` 메소드에서 이뤄진다. 이것 역시 필수는 아니지만, 덕분에 코드를 재사용한다.

### 11.5.1 `computeBatchLoss` 함수

`computeBatchLoss` 함수는 훈련 루프와 검증 루프 양쪽에서 모두 호출된다. 이름에서 알 수 있듯이, 이 함수는 샘플 배치에 대해 손실을 계산한다. 부가적으로 각 샘플에 대해 모델이 만들어내는 출력에 대한 정보도 계산해서 기록한다. 이것으로 각 클래스별로 계산이 얼마나 정확한지 백분율로 계산할 수 있으며, 분류가 잘 되지 않는 클래스를 찾아 집중 개선할 수 있다.

물론, 이 함수의 핵심 기능은 배치 단위로 모델에 입력을 넣고 손실을 계산하는 것이다. 7장에서와 같이 `CrossEntropyLoss`를 사용한다.


```python
    def computeBatchLoss(self, batch_ndx, batch_tup, batch_size, metrics_g):
        input_t, label_t, _series_list, _center_list = batch_tup
        input_t : torch.Tensor
        label_t : torch.Tensor
        
        input_g = input_t.to(self.device, non_blocking=True)
        label_g = label_t.to(self.device, non_blocking=True)

        logits_g, probability_g = self.model(input_g)
        logits_g : torch.Tensor
        probability_g : torch.Tensor

        loss_func = nn.CrossEntropyLoss(reduction="none")   
        loss_g : torch.Tensor = loss_func(
            logits_g,
            label_g[:, 1],
        )
        start_ndx = batch_ndx * batch_size
        end_ndx = start_ndx + label_t.size(0)

        metrics_g[METRICS_LABEL_NDX, start_ndx:end_ndx] = label_g[:, 1].detach()
        metrics_g[METRICS_PRED_NDX, start_ndx:end_ndx] = probability_g[:, 1].detach()
        metrics_g[METRICS_LOSS_NDX, start_ndx:end_ndx] = loss_g.detach()

        return loss_g.mean()
```

이 코드에서 유의해야 할 점은 배치 단위로 평균내는 손실값을 얻는 기본 방식을 사용하지 않는다는 것이다. 대신 손실값이 들어있는 텐서를 샘플마다 얻는다. 이렇게 하면 개별 손실값을 추적할 수 있고 원하는 방식으로(예를 들어 클래스별로) 병합할 수 있다. 지금은 각 샘플에 대한 손실 평균을 반환하여 배치 단위의 손실과 동일한 값을 넘겨준다. 샘플별로 통계값을 가질 이유가 없다면, 배치 단위의 손실값 평균 계산을 사용해도 아무 문제 없다.

여기까지 진행하면 역전파와 가중치 조정을 위해 필요한 함수 호출에 대해 할 일이 끝난 셈이 된다. 역전파 단계로 넘어가기 전에 추후 분석을 위해 샘플별 통계값을 기록한다. 이를 위해서는, 전달받은 `metircs_g` 파라미터를 사용한다.


```python
METRICS_LABEL_NDX = 0
METRICS_PRED_NDX = 1
METRICS_LOSS_NDX = 2
METRICS_SIZE = 3

    def computeBatchLoss(self, batch_ndx, batch_tup, batch_size, metrics_g):
        # ...
        start_ndx = batch_ndx * batch_size
        end_ndx = start_ndx + label_t.size(0)

        metrics_g[METRICS_LABEL_NDX, start_ndx:end_ndx] = label_g[:, 1].detach()
        metrics_g[METRICS_PRED_NDX, start_ndx:end_ndx] = probability_g[:, 1].detach()
        metrics_g[METRICS_LOSS_NDX, start_ndx:end_ndx] = loss_g.detach()

        return loss_g.mean()
```

모든 훈련(과 검증) 샘플에 대해 레이블, 예측 결과, 손실값을 기록하여 모델 동작 조사에 사용하기 위한 풍부한 세부 정보를 확보했다. 지금은 클래스 단위의 통계를 모으는 데 집중하지만, 이 정보는 나중에 이상하게 분류된 샘플을 찾아서 왜 그런지를 쉽게 확인하는 데 쓰인다. 반드시 이러한 값이 필요하지는 않으며, 일단 이런 식으로 사용 가능하다는 사실만 알아두면 좋다.

### 11.5.2 훈련 때와 유사한 검증 루프

검증 루프는 훈련 루프와 매우 비슷하지만, 조금은 단순하다. 여기서 큰 차이점은 읽기 전용이라는 점이다. 구체적으로, 손실값은 사용하지 않으며 가중치도 조정하지 않는다.

함수의 시작부터 끝까지 모델에 대해 아무것도 바꿔서는 안 된다. 또한 `with torch.no_grad()` 콘텍스트 매니저가 명시적으로 파이토치에게 기울기 계산이 필요 없다고 알려주기 때문에 훈련때보다 빠르다.

```python
    # ...
    def main(self):
        log.info(f"Starting {type(self).__name__}, {self.cli_args}")

        train_dl = self.initTrainDl()
        val_dl = self.initValDl()

        for epoch_ndx in range(1, self.cli_args.epochs + 1):

            log.info(
                "Epoch {} of {}, {}/{} batches of size {}*{}".format(
                    epoch_ndx,
                    self.cli_args.epochs,
                    len(train_dl),
                    len(val_dl),
                    self.cli_args.batch_size,
                    (torch.cuda.device_count() if self.use_cuda else 1),
                )
            )

            trnMetrics_t = self.doTraining(epoch_ndx, train_dl)
            self.logMetrics(epoch_ndx, "trn", trnMetrics_t)

            valMetrics_t = self.doValidation(epoch_ndx, val_dl)
            self.logMetrics(epoch_ndx, "val", valMetrics_t)


    def doValidation(self, epoch_ndx, val_dl):
        with torch.no_grad():
            self.model.eval()
            valMetrics_g = torch.zeros(
                METRICS_SIZE,
                len(val_dl.dataset),
                device=self.device,
            )

            val_progress = tqdm(val_dl, desc="E{} Validation".format(epoch_ndx), total=len(val_dl))
            for batch_ndx, batch_tup in enumerate(val_progress):
                self.computeBatchLoss(
                    batch_ndx, batch_tup, val_dl.batch_size, valMetrics_g
                )

        return valMetrics_g.to("cpu")
```

신경망 가중치를 조정할 필요가 없고 `computeBatchLoss` 가 넘겨주는 손실값을 사용할 일도, 옵티마이저를 참조할 필요도 없다. 그래서 루프에는 `computeBatchLoss` 호출만 남는다. `computeBatchLoss` 호출로 넘어오는 배치의 전체 손실값을 사용하지 않지만 호출의 부수효과로 `valMetrics_g`에 메트릭을 여전히 모아둔다는 사실에 주의하자.

## 11.6 성능 메트릭 출력

에포크의 마지막 작업은 성능 메트릭을 $\text{로깅}^{\text{logging}}$ 하는 것이다. 메트릭을 로깅한 후 다음 에포크의 훈련을 위해 훈련 루프의 처음으로 돌아간다. 훈련 과정을 진행할수록 결과와 진행 단계를 로깅하는 것은 중요하다. 훈련이 수렴하지 않을 수도 있고, 이를 발견하면 의미 없는 훈련에 시간을 소비하지 않도록 멈출 수도 있다. 재앙을 피하기 위해 모델이 어떻게 동작하는지 관찰하는 것은 바람직하다.

초기에 우리는 각 에포크마다 진행 상태를 로깅하기 위해 결과를 `trnMetrics_g` 와 `valMetrics_g`에 수집하고 있었다. 이제 이 두 텐서에는 훈련과 검증 단계에 필요한 모든 데이터가 포함되어 있다. 임의적인 측면은 있지만 매 에포크마다 이와 같은 작업을 수행하는 일은 흔하다.

### 11.6.1 `logMetrics` 함수

`logMetrics` 함수의 고차원 구조에 대해 이야기해보자. 함수 시그니처는 다음과 같다.

```python
    def logMetrics(
        self,
        epoch_ndx,
        mode_str,
        metrics_t,
        classificationThreshold=0.5,
    ):
```

`epoch_ndx`는 순전히 결과를 로깅할 때 표시하기 위한 용도로 사용한다. `mode_str` 인자는 메트릭이 훈련용인지 검증용인지를 나타낸다.

그리고 훈련이나 검증인 경우에 따라 `trnMetrics_t` 나 `valMetrics_t` 를 사용하는데 둘 다 `metrics_t` 파라미터로 전달된다. 두 입력 모두 `computeBatchLoss` 를 통해 만들어진 부동소수점 데이터 텐서이며, 이후 `doTraining` 이나 `doValidation` 으로부터 반환되기 직전에 CPU 영역으로 전송된다. 둘 다 세 개의 행과 (훈련 혹은 검증) 샘플 수 만큼의 열을 가진다. 여기서 세 개 행은 다음의 상수와 동일하다.

```python
METRICS_LABEL_NDX = 0
METRICS_PRED_NDX = 1
METRICS_LOSS_NDX = 2
METRICS_SIZE = 3
```


> $\textbf{마스크 구성}$

결절(양성) 샘플 또는 비결절(음성) 샘플에 대해서만 메트릭을 제한하는 마스크를 만들어보자. 그리고 클래스별 총 샘플 수와 잘 분류된 샘플의 수를 모두 세어보자. 

```python
        negLabel_mask = metrics_t[METRICS_LABEL_NDX] <= classificationThreshold   # 결절이면 label 값 1이고, clsthres=0.5보다 커지므로 False
        negPred_mask = metrics_t[METRICS_PRED_NDX] <= classificationThreshold

        posLabel_mask = ~negLabel_mask
        posPred_mask = ~negPred_mask
```

`assert` 구문은 없지만 결절의 상태 레이블은 단순하게 `True` 혹은 `False` 이므로, `metrics_t[METRICS_LABEL_NDX]`에 저장된 값은 결국 $\{0.0, 1.0\}$ 범위에 속한다. 기본값 0.5로 설정한 `classificationThreshold`와 비교하여 이진값 배열을 얻게 되면, `True`는 비결절인 경우(음성)의 레이블에 해당한다.

마찬가지로, `negPred_mask` 를 만들기 위한 비교도 수행해본다. 여기에서 `METRICS_PRED_NDX` 값은 모델이 만든 양의 예측값이며 경계값을 포함하는 0.0과 1.0 사이의 부동소수점에 해당한다. 비교 방식은 동일하지만 실젯값이 0.5에 가까워질 수도 있다. 양수 마스킹은 음수 마스킹과 정반대라고 생각하면 된다.

이제 레이블별 통계를 계산하기 위해 위에서 만든 마스크를 사용하고 `metrics_dict` 딕셔너리에 결과를 저장한다.


```python
        neg_count = int(negLabel_mask.sum())
        pos_count = int(posLabel_mask.sum())

        neg_correct = int((negLabel_mask & negPred_mask).sum())
        pos_correct = int((posLabel_mask & posPred_mask).sum())

        metrics_dict = {}
        metrics_dict["loss/all"] = metrics_t[METRICS_LOSS_NDX].mean()
        metrics_dict["loss/neg"] = metrics_t[METRICS_LOSS_NDX, negLabel_mask].mean()
        metrics_dict["loss/pos"] = metrics_t[METRICS_LOSS_NDX, posLabel_mask].mean()

        metrics_dict["correct/all"] = (
            (pos_correct + neg_correct) / np.float32(metrics_t.shape[1]) * 100
        )
        metrics_dict["correct/neg"] = neg_correct / np.float32(neg_count) * 100
        try:
            metrics_dict["correct/pos"] = pos_correct / np.float32(pos_count) * 100
        except ZeroDivisionError:
            metrics_dict["correct/pos"] = 0
```

먼저 에포크 전체에 대한 평균 손실값을 계산한다. 손실값은 훈련 중에 최소화되고 있는 단일값이므로 추적할 필요가 있다. 그리고 방금 만든 `negLabel_mask`를 사용하여 음성으로 레이블된 샘플에 대해서만 손실값의 평균을 구하도록 제한한다. 반대로 양성에 대한 손실값도 계산한다. 이런 식으로 클래스별 계산을 해두면, 특정 클래스가 다른 클래스보다 분류하기 어려운 경우를 추적하여 개선하기에 좋다.

마지막으로, 전체 샘플 중 얼마나 정확하게 맞추었는지와 각 레이블별 정확도를 계산해보자. 백분율로 출력하기 위해 비율에 100을 곱한다. 손실값과 마찬가지로, 이 값들은 개선을 위해 어디에 노력을 기울여야 할지 안내하는 역할을 한다. 계산이 끝나면, 세 번의 `log.info` 호출을 통해 결과를 로깅한다.


```python
        log.info(
            f"E{epoch_ndx} {mode_str:8} {metrics_dict['loss/all']:.4f} loss, "
            f"{metrics_dict['correct/all']:-5.1f}% correct"
        )
        log.info(
            f"E{epoch_ndx} {mode_str + '_neg':8} {metrics_dict['loss/neg']:.4f} loss, "
            f"{metrics_dict['correct/neg']:-5.1f}% correct ({neg_correct} of {neg_count})"
        )
        log.info(
            f"E{epoch_ndx} {mode_str + '_pos':8} {metrics_dict['loss/pos']:.4f} loss, "
            f"{metrics_dict['correct/pos']:-5.1f}% correct ({pos_correct} of {pos_count})"
        )

```
첫 번째 로그는 모든 샘플에 대해 계산하므로 `/log`로 태그, 음성 판별에 대한 로그는 `/neg`, 양성 판별에 대한 로그는 `/pos`로 태그되었다. 양성의 경우에 대한 로깅 구문은 `neg` 대신 `pos`를 사용하는 것 빼고는 두 번째 구문과 동일하므로 생략한다.